# Download GLORYS data from Copernicus Marine

GLORYS is the CMEMS global ocean physics reanalysis. This notebook uses the
`copernicusmarine` toolbox to subset it by region, time, depth, and variables.

Run in the `data-access-ai4ocean2026` conda env. First time only, log in once:

```bash
copernicusmarine login
```
This generates a file with credentials, by default at `~/.copernicusmarine/.copernicusmarine-credentials`.

If the env doesn't show up for jupyter, you may need to try this
```bash
conda run -n data-access-ai4ocean2026 python -m ipykernel install --user --name data-access-ai4ocean2026
```

Two steps:
1. **Download** — `download_glorys` calls `copernicusmarine.subset` and writes a NetCDF.
2. **Rechunk** — `rechunk_to_zarr` opens with `xr.open_mfdataset`, rechunks, and writes a Zarr store.

In [1]:
from download_glorys import download_glorys, rechunk_to_zarr, DEFAULT_DATASET_ID, DEFAULT_VARIABLES

ModuleNotFoundError: No module named 'copernicusmarine'

## 1. Set parameters

In [ ]:
# Updated for your explicit bounds: Lat (18N to 32N), Lon (98W to 78W)
region = dict(min_lon=-98, max_lon=-78, min_lat=18, max_lat=32, min_depth = 0, max_depth = 500)
time = dict(start_datetime="1993-01-01", end_datetime="2004-12-31")
variables = DEFAULT_VARIABLES

## 2. Peek at the dataset (metadata only, no download)

In [29]:
import copernicusmarine

ds = copernicusmarine.open_dataset(
    dataset_id=DEFAULT_DATASET_ID,
    variables=variables,
    minimum_longitude=region["min_lon"],
    maximum_longitude=region["max_lon"],
    minimum_latitude=region["min_lat"],
    maximum_latitude=region["max_lat"],
    start_datetime=time["start_datetime"],
    end_datetime=time["end_datetime"],
)
ds

INFO - 2026-07-23T22:37:08Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username:  dapte
Copernicus Marine password:  ········


INFO - 2026-07-23T22:37:17Z - Selected dataset version: "202311"
INFO - 2026-07-23T22:37:17Z - Selected dataset part: "default"


<xarray.Dataset> Size: 12GB
Dimensions:    (depth: 50, latitude: 169, longitude: 121, time: 366)
Coordinates:
  * depth      (depth) float32 200B 0.494 1.541 2.646 ... 5.275e+03 5.728e+03
  * latitude   (latitude) float32 676B 18.0 18.08 18.17 ... 31.83 31.92 32.0
  * longitude  (longitude) float32 484B -88.0 -87.92 -87.83 ... -78.08 -78.0
  * time       (time) datetime64[ns] 3kB 2004-01-01 2004-01-02 ... 2004-12-31
Data variables:
    thetao     (time, depth, latitude, longitude) float64 3GB dask.array<chunksize=(366, 50, 32, 16), meta=np.ndarray>
    so         (time, depth, latitude, longitude) float64 3GB dask.array<chunksize=(366, 50, 32, 16), meta=np.ndarray>
    uo         (time, depth, latitude, longitude) float64 3GB dask.array<chunksize=(366, 50, 32, 16), meta=np.ndarray>
    vo         (time, depth, latitude, longitude) float64 3GB dask.array<chunksize=(366, 50, 32, 16), meta=np.ndarray>
    zos        (time, latitude, longitude) float64 60MB dask.array<chunksize=(366, 32, 16), meta=np.ndarray>
Attributes: (12/25)
    Conventions:               CF-1.4
    bulletin_date:             2021-07-07 00:00:00
    bulletin_type:             operational
    comment:                   CMEMS product
    domain_name:               GL12
    easting:                   longitude
    ...                        ...
    references:                http://www.mercator-ocean.fr
    source:                    MERCATOR GLORYS12V1
    title:                     daily mean fields from Global Ocean Physics An...
    z_max:                     5727.9169921875
    z_min:                     0.49402499198913574
    copernicusmarine_version:  2.4.1

## 3. Step 1 — Download to NetCDF

In [31]:
nc_path = download_glorys(
    variables=variables,
    **region,
    **time,
    output_filename="glorys_gom_jan2004.nc",
    output_dir="data",
)
nc_path

INFO - 2026-07-23T22:44:08Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username:  dapte
Copernicus Marine password:  ········


INFO - 2026-07-23T22:44:17Z - Selected dataset version: "202311"
INFO - 2026-07-23T22:44:17Z - Selected dataset part: "default"


  0%|          | [00:00<?]

INFO - 2026-07-23T23:00:55Z - Total size of the download: 3.49 GB.


Downloaded data/glorys_gom_jan2004.nc


'data/glorys_gom_jan2004.nc'

## 4. Step 2 — Rechunk to Zarr

In [32]:
rechunk_to_zarr(
    input_path=nc_path,
    output_path="data/glorys_gom_jan2004.zarr",
    chunks={"time": 1, "latitude": -1, "longitude": -1},
)

/Users/dhruvgirishapte/Documents/analog-forecasting-ai4ocean2026/data-access-env/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Wrote data/glorys_gom_jan2004.zarr


### With dask parallel I/O (useful for large files / many input files)

In [33]:
from dask.distributed import Client

client = Client()  # open client.dashboard_link to watch progress
client

Python(70184) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(70185) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(70186) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(70187) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 8.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:61582,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:61598,Total threads: 2
Dashboard: http://127.0.0.1:61602/status,Memory: 2.00 GiB
Nanny: tcp://127.0.0.1:61585,


## 5. Read the Zarr store back

In [34]:
import xarray as xr

ds_zarr = xr.open_zarr("data/glorys_gom_jan2004.zarr")
ds_zarr

<xarray.Dataset> Size: 12GB
Dimensions:    (time: 366, depth: 50, latitude: 169, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 3kB 2004-01-01 2004-01-02 ... 2004-12-31
  * depth      (depth) float32 200B 0.494 1.541 2.646 ... 5.275e+03 5.728e+03
  * latitude   (latitude) float32 676B 18.0 18.08 18.17 ... 31.83 31.92 32.0
  * longitude  (longitude) float32 484B -88.0 -87.92 -87.83 ... -78.08 -78.0
Data variables:
    so         (time, depth, latitude, longitude) float64 3GB dask.array<chunksize=(1, 50, 169, 121), meta=np.ndarray>
    thetao     (time, depth, latitude, longitude) float64 3GB dask.array<chunksize=(1, 50, 169, 121), meta=np.ndarray>
    uo         (time, depth, latitude, longitude) float64 3GB dask.array<chunksize=(1, 50, 169, 121), meta=np.ndarray>
    vo         (time, depth, latitude, longitude) float64 3GB dask.array<chunksize=(1, 50, 169, 121), meta=np.ndarray>
    zos        (time, latitude, longitude) float64 60MB dask.array<chunksize=(1, 169, 121), meta=np.ndarray>
Attributes: (12/25)
    Conventions:               CF-1.4
    bulletin_date:             2021-07-07 00:00:00
    bulletin_type:             operational
    comment:                   CMEMS product
    domain_name:               GL12
    easting:                   longitude
    ...                        ...
    references:                http://www.mercator-ocean.fr
    source:                    MERCATOR GLORYS12V1
    title:                     daily mean fields from Global Ocean Physics An...
    z_max:                     5727.9169921875
    z_min:                     0.49402499198913574
    copernicusmarine_version:  2.4.1